# 📊 Day 03a: CPU Scheduling Algorithms

---

## 🎯 What You'll Master Today
- FCFS, SJF, SRTF, Round Robin, Priority Scheduling
- Gantt charts and metric calculations (turnaround, waiting, response time)
- Preemptive vs non-preemptive, starvation, convoy effect

---

## 🎭 The Analogy

**CPU scheduling** = managing a single billing counter at a store  
- FCFS = first in line gets served first (fair but slow if big order ahead)  
- SJF = person with fewest items goes first (fast average, unfair to big carts)  
- Round Robin = everyone gets 2 minutes, then goes to back of line  

---

```
╔══════════════════════════════════════════════════════════╗
║              KEY SCHEDULING METRICS                      ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Turnaround Time = Completion - Arrival                  ║
║  Waiting Time    = Turnaround - Burst                    ║
║  Response Time   = First execution - Arrival             ║
║                                                          ║
║  Goal: minimize average waiting & turnaround time        ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# Helper: Display scheduling results
# ============================================

def display_results(name, processes, completion, burst, arrival):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  {'PID':<6} {'Arrival':<9} {'Burst':<8} {'Compl':<8} {'TAT':<8} {'Wait':<8}")
    print(f"  {'-'*48}")
    
    total_tat = total_wt = 0
    for pid in processes:
        tat = completion[pid] - arrival[pid]
        wt = tat - burst[pid]
        total_tat += tat
        total_wt += wt
        print(f"  P{pid:<5} {arrival[pid]:<9} {burst[pid]:<8} {completion[pid]:<8} {tat:<8} {wt:<8}")
    
    n = len(processes)
    print(f"\n  Avg TAT: {total_tat/n:.2f}  |  Avg Waiting: {total_wt/n:.2f}")

In [ ]:
# ============================================
# 1. FCFS — First Come First Served
# ============================================

def fcfs(processes, arrival, burst):
    """Non-preemptive. Serve in arrival order."""
    # Sort by arrival time
    order = sorted(processes, key=lambda p: arrival[p])
    
    completion = {}
    time = 0
    gantt = []
    
    for pid in order:
        if time < arrival[pid]:
            time = arrival[pid]  # CPU idle
        gantt.append((pid, time, time + burst[pid]))
        time += burst[pid]
        completion[pid] = time
    
    # Print Gantt chart
    print("  Gantt: ", end="")
    for pid, start, end in gantt:
        print(f"| P{pid}({start}-{end}) ", end="")
    print("|")
    
    return completion


# Example
processes = [1, 2, 3, 4]
arrival =   {1: 0, 2: 1, 3: 2, 4: 3}
burst =     {1: 8, 2: 4, 3: 9, 4: 5}

comp = fcfs(processes, arrival, burst)
display_results("FCFS (First Come First Served)", processes, comp, burst, arrival)

In [ ]:
# ============================================
# 2. SJF — Shortest Job First (Non-Preemptive)
# ============================================

def sjf(processes, arrival, burst):
    """Non-preemptive. At each decision point, pick shortest burst."""
    remaining = set(processes)
    completion = {}
    time = 0
    gantt = []
    
    while remaining:
        # Available processes (arrived and not done)
        available = [p for p in remaining if arrival[p] <= time]
        
        if not available:
            time = min(arrival[p] for p in remaining)
            continue
        
        # Pick shortest burst
        pid = min(available, key=lambda p: burst[p])
        gantt.append((pid, time, time + burst[pid]))
        time += burst[pid]
        completion[pid] = time
        remaining.remove(pid)
    
    print("  Gantt: ", end="")
    for pid, start, end in gantt:
        print(f"| P{pid}({start}-{end}) ", end="")
    print("|")
    
    return completion


comp = sjf(processes, arrival, burst)
display_results("SJF (Shortest Job First — Non-Preemptive)", processes, comp, burst, arrival)

In [ ]:
# ============================================
# 3. SRTF — Shortest Remaining Time First (Preemptive SJF)
# ============================================

def srtf(processes, arrival, burst):
    """Preemptive. At each unit time, pick process with shortest remaining time."""
    remaining_burst = {p: burst[p] for p in processes}
    completion = {}
    time = 0
    done = 0
    n = len(processes)
    
    while done < n:
        available = [p for p in processes if arrival[p] <= time and remaining_burst[p] > 0]
        
        if not available:
            time += 1
            continue
        
        pid = min(available, key=lambda p: remaining_burst[p])
        remaining_burst[pid] -= 1
        time += 1
        
        if remaining_burst[pid] == 0:
            completion[pid] = time
            done += 1
    
    return completion


comp = srtf(processes, arrival, burst)
display_results("SRTF (Shortest Remaining Time First — Preemptive)", processes, comp, burst, arrival)

In [ ]:
# ============================================
# 4. Round Robin
# ============================================
from collections import deque

def round_robin(processes, arrival, burst, quantum: int):
    """Preemptive. Each process gets 'quantum' units, then back of queue."""
    remaining_burst = {p: burst[p] for p in processes}
    completion = {}
    time = 0
    queue = deque()
    arrived = set()
    gantt = []
    
    # Sort by arrival
    sorted_procs = sorted(processes, key=lambda p: arrival[p])
    idx = 0
    
    # Add first process
    while idx < len(sorted_procs) and arrival[sorted_procs[idx]] <= time:
        queue.append(sorted_procs[idx])
        arrived.add(sorted_procs[idx])
        idx += 1
    
    while queue:
        pid = queue.popleft()
        exec_time = min(quantum, remaining_burst[pid])
        gantt.append((pid, time, time + exec_time))
        
        time += exec_time
        remaining_burst[pid] -= exec_time
        
        # Add newly arrived processes
        while idx < len(sorted_procs) and arrival[sorted_procs[idx]] <= time:
            queue.append(sorted_procs[idx])
            arrived.add(sorted_procs[idx])
            idx += 1
        
        if remaining_burst[pid] > 0:
            queue.append(pid)  # back of queue
        else:
            completion[pid] = time
        
        if not queue and idx < len(sorted_procs):
            time = arrival[sorted_procs[idx]]
            queue.append(sorted_procs[idx])
            arrived.add(sorted_procs[idx])
            idx += 1
    
    print("  Gantt: ", end="")
    for pid, start, end in gantt:
        print(f"| P{pid}({start}-{end}) ", end="")
    print("|")
    
    return completion


comp = round_robin(processes, arrival, burst, quantum=4)
display_results("Round Robin (Quantum = 4)", processes, comp, burst, arrival)

```
╔══════════════════════════════════════════════════════════════╗
║            SCHEDULING ALGORITHMS COMPARISON                  ║
╠═══════════════════╦══════════════════════════════════════════╣
║  Algorithm         ║  Key Properties                        ║
╠═══════════════════╬══════════════════════════════════════════╣
║  FCFS              ║  Simple, convoy effect, non-preemptive ║
║  SJF               ║  Optimal avg wait, starvation risk     ║
║  SRTF              ║  Preemptive SJF, best avg wait         ║
║  Round Robin       ║  Fair, time quantum matters, preemptive║
║  Priority          ║  Priority-based, starvation → aging    ║
║  MLFQ              ║  Adaptive, real OS use (Linux CFS)     ║
╚═══════════════════╩══════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Convoy effect? | Short processes stuck behind long one in FCFS |
| 2 | SJF starvation? | Long processes may never execute if short ones keep arriving. Fix: aging |
| 3 | Optimal quantum for RR? | Too small → overhead. Too large → becomes FCFS. Usually 10-100ms |
| 4 | Preemptive vs non-preemptive? | Preemptive can interrupt running process. Non-preemptive waits until burst completes |
| 5 | What does Linux use? | CFS (Completely Fair Scheduler) — weighted fair queuing, O(log n) with red-black tree |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • FCFS = simple but convoy effect                       │
## │  • SJF/SRTF = optimal avg wait but can starve           │
## │  • Round Robin = fair with time quantum                  │
## │  • TAT = Completion - Arrival                            │
## │  • WT  = TAT - Burst                                    │
## │  • Know how to draw Gantt charts and calculate metrics   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **IPC** — how processes communicate with each other